# 6. StyleVAR Generalization Test on Our Content/Style Images

This notebook tests the official StyleVAR model on our own content/style folders.

StyleVAR is not text-to-image. It uses:

```text
content image + style reference image -> stylized output
```

The goal here is not to improve StyleVAR yet. The goal is to verify the paper's limitation discussion with our own images, while avoiding any human/portrait-related cases.

We test:

```text
1. Easier non-human cases: landscapes, architecture, clear animals/objects.
2. Harder non-human cases: semantic mismatch, abstract style, strong domain shift, ambiguous natural content.
3. SFT vs GRPO: whether reward tuning improves perceptual style/content trade-off.
```

Official paper/repo:

```text
paper: https://arxiv.org/pdf/2604.21052
repo : https://github.com/Senfier-LiqiJing/StyleVAR
ckpt : https://huggingface.co/Senfier-LiqiJing/StyleVAR
```


## 0. Runtime Setup

Use a GPU runtime. StyleVAR is heavier than our VAR-CLIP diagnostics, so if Colab T4 is unstable, try Kaggle GPU, L4, A100, or Colab Pro.

The setup keeps the runtime PyTorch version instead of installing the paper's pinned PyTorch version. This avoids breaking Colab/Kaggle CUDA.


In [ ]:
!nvidia-smi

import os
import sys
import math
import json
import random
import subprocess
from pathlib import Path

if Path('/kaggle/working').exists():
    RUNTIME_ROOT = Path('/kaggle/working')
elif Path('/content').exists():
    RUNTIME_ROOT = Path('/content')
else:
    RUNTIME_ROOT = Path.cwd()

WORKSPACE_REPO = 'https://github.com/LeeHoang2710/Style-Transfer-Experiment.git'
WORKSPACE = RUNTIME_ROOT / 'VAR_Style_Transfer_Workspace'
STYLEVAR_REPO = 'https://github.com/Senfier-LiqiJing/StyleVAR.git'
STYLEVAR_DIR = RUNTIME_ROOT / 'StyleVAR'
OUTPUT_DIR = RUNTIME_ROOT / 'StyleVAR_outputs' / 'generalization_test'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('runtime root:', RUNTIME_ROOT)
print('outputs:', OUTPUT_DIR)


In [ ]:
# Clone or refresh our experiment workspace.
if not WORKSPACE.exists():
    subprocess.run(['git', 'clone', '--depth', '1', WORKSPACE_REPO, str(WORKSPACE)], check=True)
else:
    subprocess.run(['git', '-C', str(WORKSPACE), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(WORKSPACE), 'reset', '--hard', 'origin/main'], check=True)

# Clone or refresh official StyleVAR.
if not STYLEVAR_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', STYLEVAR_REPO, str(STYLEVAR_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(STYLEVAR_DIR), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(STYLEVAR_DIR), 'reset', '--hard', 'origin/main'], check=True)

CONTENT_DIR = WORKSPACE / 'content'
STYLE_DIR = WORKSPACE / 'style'
assert CONTENT_DIR.exists(), CONTENT_DIR
assert STYLE_DIR.exists(), STYLE_DIR

os.chdir(STYLEVAR_DIR)
if str(STYLEVAR_DIR) not in sys.path:
    sys.path.insert(0, str(STYLEVAR_DIR))

print('workspace:', WORKSPACE)
print('StyleVAR source:', STYLEVAR_DIR)
print('content images:', len(list(CONTENT_DIR.glob('*.png'))))
print('style categories:', len([p for p in STYLE_DIR.iterdir() if p.is_dir()]))

# The public StyleVAR code imports a project-local `dist` helper, but the file is absent in the repo.
# For this notebook we only need single-GPU inference, so this shim provides the small API used by the model.
DIST_SHIM = STYLEVAR_DIR / 'dist.py'
DIST_SHIM.write_text("""
import torch
from torch import distributed as tdist


def initialized():
    return tdist.is_available() and tdist.is_initialized()


def get_device():
    return torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


def get_rank():
    return tdist.get_rank() if initialized() else 0


def get_world_size():
    return tdist.get_world_size() if initialized() else 1


def is_master():
    return get_rank() == 0


def barrier():
    if initialized():
        tdist.barrier()


def allreduce(x):
    if initialized():
        tdist.all_reduce(x)
    return x


def all_reduce(x):
    return allreduce(x)


def broadcast(x, src=0):
    if initialized():
        tdist.broadcast(x, src=src)
    return x
""")

# Make sure Python sees the freshly-created dist.py if this cell is rerun.
if 'dist' in sys.modules:
    del sys.modules['dist']
print('single-GPU dist shim:', DIST_SHIM)


In [ ]:
# Install only the lightweight dependencies we need for inference.
# Do not install the repo requirements.txt directly because it pins a full CUDA/PyTorch stack.
!pip -q install huggingface_hub einops safetensors pyyaml typed-argument-parser tqdm pandas matplotlib pillow scipy lpips

import gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision
from PIL import Image, ImageOps
from torchvision import transforms
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
assert torch.cuda.is_available(), 'StyleVAR inference needs a CUDA GPU runtime.'


## 1. Download Official StyleVAR Checkpoints

Default mode downloads only:

```text
VAE  : tokenizer, required
GRPO : merged StyleVAR model, required
```

We do **not** download the 9GB SFT checkpoint by default because normal Colab RAM can die while handling it. The Hugging Face model card says the GRPO checkpoint is already merged into the base model, so it can be loaded directly like a normal StyleVAR checkpoint.

Set `DOWNLOAD_SFT = True` only on a high-RAM runtime if you really need SFT-vs-GRPO comparison.


In [ ]:
CKPT_DIR = STYLEVAR_DIR / 'ckpt'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
HF_REPO = 'Senfier-LiqiJing/StyleVAR'

DOWNLOAD_SFT = False  # Keep False on normal Colab. The SFT checkpoint is about 9GB.

CHECKPOINT_CANDIDATES = {
    'vae': ['vae_ch160v4096z32.pth'],
    # HF/model card may show hyphen while the file listing may use underscore, so keep both.
    'grpo': ['StyleVAR_GRPO.pth', 'StyleVAR-GRPO.pth'],
}
if DOWNLOAD_SFT:
    CHECKPOINT_CANDIDATES['sft'] = ['StyleVAR_SFT.pth']

ckpt_paths = {}
for name, filenames in CHECKPOINT_CANDIDATES.items():
    last_error = None
    for filename in filenames:
        try:
            print('downloading/checking:', filename)
            ckpt_paths[name] = Path(hf_hub_download(repo_id=HF_REPO, filename=filename, local_dir=CKPT_DIR))
            print(name, ckpt_paths[name], ckpt_paths[name].stat().st_size / 1e6, 'MB')
            break
        except Exception as exc:
            last_error = exc
            print(f'could not download {filename}: {type(exc).__name__}: {exc}')
    if name not in ckpt_paths:
        raise last_error

print('available checkpoints:', ckpt_paths)


## 2. Load StyleVAR

This uses the same configuration as the official `eval/infer_grpo.py` script:

```text
VAR depth: 20
patch scales: 1,2,3,4,5,6,8,10,13,16
VQ codebook: 4096
latent channels: 32
style encoder dim: 512
```

Important memory rule:

```text
GRPO is loaded directly.
SFT is optional and disabled by default.
```

That avoids the normal-Colab failure mode where the runtime dies while handling the huge SFT checkpoint.


In [ ]:
from models import build_vae_stylevar

# Optional LoRA utilities. Some released checkpoints may be merged; some may be adapter-style.
try:
    from utils.lora import apply_lora, set_lora_enabled
except Exception as exc:
    apply_lora = None
    set_lora_enabled = None
    print('LoRA utilities unavailable:', repr(exc))

# Avoid unnecessary default initialization before loading checkpoints.
setattr(torch.nn.Linear, 'reset_parameters', lambda self: None)
setattr(torch.nn.LayerNorm, 'reset_parameters', lambda self: None)

device = torch.device('cuda:0')
PATCH_NUMS = tuple(int(x) for x in '1_2_3_4_5_6_8_10_13_16'.split('_'))

vae, stylevar = build_vae_stylevar(
    device=device,
    patch_nums=PATCH_NUMS,
    V=4096,
    Cvae=32,
    ch=160,
    share_quant_resi=4,
    depth=20,
    shared_aln=False,
    attn_l2_norm=True,
    flash_if_available=True,
    fused_if_available=True,
    init_adaln=0.5,
    init_adaln_gamma=1e-5,
    init_head=0.02,
    init_std=-1,
    style_enc_dim=512,
)

vae.load_state_dict(torch.load(ckpt_paths['vae'], map_location='cpu'), strict=True)
vae.eval()
stylevar.eval()
for model in (vae, stylevar):
    for parameter in model.parameters():
        parameter.requires_grad_(False)

print('StyleVAR model object is built. Next cell loads SFT/GRPO weights before generation.')


In [ ]:
def safe_torch_load(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')


def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        if 'trainer' in checkpoint and isinstance(checkpoint['trainer'], dict) and 'var_wo_ddp' in checkpoint['trainer']:
            return checkpoint['trainer']['var_wo_ddp']
        if 'model' in checkpoint and isinstance(checkpoint['model'], dict):
            return checkpoint['model']
        if 'state_dict' in checkpoint and isinstance(checkpoint['state_dict'], dict):
            return checkpoint['state_dict']
    return checkpoint


def looks_like_lora_state(state):
    if not isinstance(state, dict):
        return False
    keys = list(state.keys())
    return any(('lora' in key.lower()) or ('adapter' in key.lower()) for key in keys)


def load_state_with_report(model, state, label, strict=False):
    missing, unexpected = model.load_state_dict(state, strict=strict)
    print(f'[{label}] load_state_dict(strict={strict}) | missing={len(missing)} unexpected={len(unexpected)}')
    if missing and len(missing) <= 8:
        print('missing:', missing)
    if unexpected and len(unexpected) <= 8:
        print('unexpected:', unexpected)
    return missing, unexpected


def load_stylevar_weights(model, ckpt_path, label):
    checkpoint = safe_torch_load(ckpt_path)
    state = extract_state_dict(checkpoint)

    if looks_like_lora_state(state):
        if 'sft' not in ckpt_paths:
            raise RuntimeError('This GRPO checkpoint looks like LoRA, but SFT was not downloaded. Set DOWNLOAD_SFT=True and rerun checkpoint download.')
        if apply_lora is None:
            raise RuntimeError('Checkpoint looks like LoRA, but LoRA utilities are unavailable.')
        sft_checkpoint = safe_torch_load(ckpt_paths['sft'])
        sft_state = extract_state_dict(sft_checkpoint)
        load_state_with_report(model, sft_state, 'SFT base for LoRA', strict=True)
        del sft_state, sft_checkpoint
        clear_memory()
        args = checkpoint.get('args', {}) if isinstance(checkpoint, dict) else {}
        rank = args.get('lora_rank', 256)
        alpha = args.get('lora_alpha', 512.0)
        print(f'[{label}] applying LoRA adapters rank={rank}, alpha={alpha}')
        apply_lora(model, rank, alpha)
        merged_state = model.state_dict()
        merged_state.update(state)
        load_state_with_report(model, merged_state, f'{label} LoRA merged into SFT', strict=False)
        del state, merged_state, checkpoint
        clear_memory()
        if set_lora_enabled is not None:
            set_lora_enabled(model, True)
        model.eval()
        return {'checkpoint': None, 'mode': 'sft_plus_lora'}

    # Current released SFT/GRPO checkpoints are full merged model states under the `model` key.
    load_state_with_report(model, state, label.upper(), strict=True)
    del state, checkpoint
    clear_memory()
    if set_lora_enabled is not None:
        try:
            set_lora_enabled(model, False)
        except Exception:
            pass
    model.eval()
    return {'checkpoint': None, 'mode': 'full_merged'}


def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()


## 3. Define Non-Human Test Pairs

No human, no portrait content, and no obviously human portrait style references.

The cases are split by expected generalization behavior:

```text
easier: landscape, architecture, clear animals/objects
harder: semantic mismatch, abstract style, strong domain shift, ambiguous natural content
```

You can edit this list freely. The notebook checks every path before running.


In [ ]:
TEST_CASES = [
    # Expected easier: natural scene / architecture / clear animal.
    {
        'case_id': 'easy_lake_monet_landscape',
        'group': 'expected_easier_landscape',
        'content_path': CONTENT_DIR / '0014.png',
        'style_path': STYLE_DIR / 'Monet' / 'Monet001.png',
        'note': 'Landscape content with landscape painting style. This should be one of the better cases.',
    },
    {
        'case_id': 'easy_mountain_watercolor_landscape',
        'group': 'expected_easier_landscape',
        'content_path': CONTENT_DIR / '0011.png',
        'style_path': STYLE_DIR / 'WaterColor' / 'WC001.png',
        'note': 'Mountain/landscape style transfer, close to the paper examples.',
    },
    {
        'case_id': 'easy_house_watercolor_architecture',
        'group': 'expected_easier_architecture',
        'content_path': CONTENT_DIR / '0015.png',
        'style_path': STYLE_DIR / 'WaterColor' / 'WC002.png',
        'note': 'Architecture with architecture-like watercolor style.',
    },
    {
        'case_id': 'easy_dog_sketch',
        'group': 'expected_easier_animal',
        'content_path': CONTENT_DIR / '0003.png',
        'style_path': STYLE_DIR / 'Sketch' / 'S005.png',
        'note': 'Dog content with same-domain animal sketch style.',
    },
    {
        'case_id': 'easy_elephant_line_drawing',
        'group': 'expected_easier_animal',
        'content_path': CONTENT_DIR / '0010.png',
        'style_path': STYLE_DIR / 'LineDrawing' / 'LD001.png',
        'note': 'Elephant with simple line drawing. Checks whether structure survives sparse style.',
    },
    {
        'case_id': 'easy_cat_pixel_art',
        'group': 'expected_easier_animal',
        'content_path': CONTENT_DIR / '0001.png',
        'style_path': STYLE_DIR / 'PixelArt' / 'PA012.png',
        'note': 'Cat with animal-like pixel/mosaic reference.',
    },
    {
        'case_id': 'easy_cat_watercolor',
        'group': 'expected_easier_animal',
        'content_path': CONTENT_DIR / '0006.png',
        'style_path': STYLE_DIR / 'WaterColor' / 'WC003.png',
        'note': 'Cat face with soft watercolor style. Tests animal structure plus gentle texture.',
    },
    {
        'case_id': 'easy_lake_oil_painting',
        'group': 'expected_easier_landscape',
        'content_path': CONTENT_DIR / '0014.png',
        'style_path': STYLE_DIR / 'OilPainting' / 'OP005.png',
        'note': 'Lake/landscape with oil-painting landscape style.',
    },

    # Expected harder but still non-human: mismatch, abstraction, domain shift, ambiguous content.
    {
        'case_id': 'hard_cat_abstract',
        'group': 'expected_harder_semantic_mismatch',
        'content_path': CONTENT_DIR / '0006.png',
        'style_path': STYLE_DIR / 'Abstract' / 'AB002.png',
        'note': 'Animal content with abstract style. Tests content destruction risk without using humans.',
    },
    {
        'case_id': 'hard_clouds_anime_sky',
        'group': 'expected_harder_ambiguous_content',
        'content_path': CONTENT_DIR / '0004.png',
        'style_path': STYLE_DIR / 'AnimeStyle' / 'AS007.png',
        'note': 'Ambiguous natural content with anime style. Checks if StyleVAR hallucinates structure.',
    },
    {
        'case_id': 'hard_lake_cyberpunk',
        'group': 'expected_harder_style_domain_shift',
        'content_path': CONTENT_DIR / '0014.png',
        'style_path': STYLE_DIR / 'Cyberpunk' / 'Cyberpunk004.png',
        'note': 'Natural landscape with neon cyberpunk style. Tests strong domain shift.',
    },
    {
        'case_id': 'hard_house_surrealism',
        'group': 'expected_harder_style_domain_shift',
        'content_path': CONTENT_DIR / '0015.png',
        'style_path': STYLE_DIR / 'Surrealism' / 'S001.png',
        'note': 'Architecture with surreal style. Tests whether layout gets warped.',
    },
    {
        'case_id': 'hard_elephant_abstract',
        'group': 'expected_harder_semantic_mismatch',
        'content_path': CONTENT_DIR / '0010.png',
        'style_path': STYLE_DIR / 'Abstract' / 'AB001.png',
        'note': 'Large animal with abstract style. Tests whether coarse structure is preserved.',
    },
    {
        'case_id': 'hard_house_cyberpunk',
        'group': 'expected_harder_style_domain_shift',
        'content_path': CONTENT_DIR / '0015.png',
        'style_path': STYLE_DIR / 'Cyberpunk' / 'Cyberpunk001.png',
        'note': 'House with cyberpunk reference. Good non-human domain-shift stress test.',
    },
]

for case in TEST_CASES:
    assert case['content_path'].exists(), case['content_path']
    assert case['style_path'].exists(), case['style_path']
    combined_text = (case['case_id'] + ' ' + case['group'] + ' ' + case['note']).lower()
    assert 'portrait' not in combined_text, case['case_id']

pd.DataFrame([{**case, 'content_path': str(case['content_path']), 'style_path': str(case['style_path'])} for case in TEST_CASES])


In [ ]:
def pil_to_display(path, size=256):
    return ImageOps.fit(Image.open(path).convert('RGB'), (size, size), method=Image.Resampling.LANCZOS)


def show_case_preview(cases, ncols=4):
    rows = len(cases)
    plt.figure(figsize=(4 * ncols, 3.8 * rows))
    for row, case in enumerate(cases):
        for col, key in enumerate(['content_path', 'style_path']):
            plt.subplot(rows, ncols, row * ncols + col + 1)
            plt.imshow(pil_to_display(case[key]))
            title = 'content' if key == 'content_path' else 'style'
            plt.title(f'{case["case_id"]}\n{title}', fontsize=9)
            plt.axis('off')
        plt.subplot(rows, ncols, row * ncols + 3)
        plt.text(0.0, 0.5, case['group'], fontsize=10, wrap=True)
        plt.axis('off')
        plt.subplot(rows, ncols, row * ncols + 4)
        plt.text(0.0, 0.5, case['note'], fontsize=9, wrap=True)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

show_case_preview(TEST_CASES)


## 4. Inference Helpers

Each result is cached as a PNG. If the runtime disconnects, rerun the notebook and it skips images that already exist.


In [ ]:
IMAGE_SIZE = 256
TOP_K = 900
TOP_P = 0.96
SEED = 42

# Normal Colab RAM cannot safely handle the huge SFT checkpoint.
# Use GRPO only first. It is the paper's final model and is already merged.
RUN_SFT = False
RUN_GRPO = True
MAX_CASES_TO_RUN = 4  # Start small. Increase after the first grid succeeds.

image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3),
])


def load_condition_image(path):
    return image_transform(Image.open(path).convert('RGB')).unsqueeze(0).to(device)


def save_tensor_image_01(image_01, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torchvision.utils.save_image(image_01.detach().float().cpu().clamp(0, 1), path)


def read_tensor_image_01(path):
    return transforms.functional.to_tensor(Image.open(path).convert('RGB')).unsqueeze(0)


@torch.no_grad()
def run_one_stylevar_case(model, case, seed=SEED, top_k=TOP_K, top_p=TOP_P):
    content = load_condition_image(case['content_path'])
    style = load_condition_image(case['style_path'])
    generated = model.autoregressive_infer(
        B=1,
        style_img=style,
        content_img=content,
        top_k=top_k,
        top_p=top_p,
        g_seed=seed,
    )
    del content, style
    clear_memory()
    return generated.clamp(0, 1)


def generate_checkpoint_outputs(label, checkpoint_path, cases):
    load_stylevar_weights(stylevar, checkpoint_path, label)
    stylevar.eval()

    records = []
    active_cases = cases[:MAX_CASES_TO_RUN] if MAX_CASES_TO_RUN is not None else cases
    for case in tqdm(active_cases, desc=f'Generating {label}'):
        out_path = OUTPUT_DIR / label / f'{case["case_id"]}.png'
        if not out_path.exists():
            result = run_one_stylevar_case(stylevar, case)
            save_tensor_image_01(result, out_path)
            del result
            clear_memory()
        records.append({
            'model': label,
            'case_id': case['case_id'],
            'group': case['group'],
            'content_path': str(case['content_path']),
            'style_path': str(case['style_path']),
            'output_path': str(out_path),
            'note': case['note'],
        })
    return pd.DataFrame(records)


## 5. Run Inference

Run this cell once.

Default safe mode:

```text
GRPO only
first 4 curated cases
no 9GB SFT checkpoint
```

After this succeeds, increase `MAX_CASES_TO_RUN` or set it to `None`.


In [ ]:
all_records = []

if RUN_SFT:
    if 'sft' not in ckpt_paths:
        raise RuntimeError('RUN_SFT=True but SFT checkpoint was not downloaded. Set DOWNLOAD_SFT=True in the checkpoint cell.')
    sft_records = generate_checkpoint_outputs('sft', ckpt_paths['sft'], TEST_CASES)
    all_records.append(sft_records)

if RUN_GRPO:
    grpo_records = generate_checkpoint_outputs('grpo', ckpt_paths['grpo'], TEST_CASES)
    all_records.append(grpo_records)

clear_memory()
result_table = pd.concat(all_records, ignore_index=True) if all_records else pd.DataFrame()
result_table.to_csv(OUTPUT_DIR / 'stylevar_generalization_outputs.csv', index=False)
display(result_table)
print('saved:', OUTPUT_DIR / 'stylevar_generalization_outputs.csv')


## 6. Visual Comparison Grid

Look for these non-human failure modes:

```text
content loss       : object/scene layout changes too much
style weakness     : output still looks like plain content image
style overcopying  : style image structure leaks into output
OOD hallucination  : new objects appear that are not in content/style
abstract collapse  : abstract style destroys recognizable content
```


In [ ]:
def make_comparison_grid(cases, save_name='stylevar_sft_grpo_comparison.png'):
    columns = ['content', 'style']
    if RUN_SFT:
        columns.append('sft')
    if RUN_GRPO:
        columns.append('grpo')

    rows = len(cases)
    plt.figure(figsize=(4 * len(columns), 4 * rows))
    for row, case in enumerate(cases):
        images = [
            ('content', case['content_path']),
            ('style', case['style_path']),
        ]
        if RUN_SFT:
            images.append(('SFT', OUTPUT_DIR / 'sft' / f'{case["case_id"]}.png'))
        if RUN_GRPO:
            images.append(('GRPO', OUTPUT_DIR / 'grpo' / f'{case["case_id"]}.png'))

        for col, (title, path) in enumerate(images):
            plt.subplot(rows, len(columns), row * len(columns) + col + 1)
            plt.imshow(pil_to_display(path))
            if row == 0:
                plt.title(title, fontsize=12)
            if col == 0:
                plt.ylabel(case['case_id'] + '\n' + case['group'], fontsize=8)
            plt.xticks([])
            plt.yticks([])
    plt.tight_layout()
    out_path = OUTPUT_DIR / save_name
    plt.savefig(out_path, dpi=180, bbox_inches='tight')
    plt.show()
    print('saved:', out_path)

make_comparison_grid(TEST_CASES)


In [ ]:
# Smaller separate sheets are easier to inspect than one very tall grid.
easy_cases = [case for case in TEST_CASES if 'expected_easier' in case['group']]
hard_cases = [case for case in TEST_CASES if 'expected_harder' in case['group']]

make_comparison_grid(easy_cases, save_name='stylevar_expected_easier_cases.png')
make_comparison_grid(hard_cases, save_name='stylevar_expected_harder_cases.png')


## 7. Manual Evaluation Sheet

The paper reports automatic metrics, but for our research decision the first pass should be visual and diagnostic.

Use this scoring idea:

```text
content_score: 1 bad, 5 content preserved well
style_score  : 1 weak/no style, 5 style clearly transferred
quality_score: 1 broken, 5 coherent image
failure_tags : content_loss, style_weak, style_overcopy, face_failure, hallucination, good
```

After you inspect the grids, fill the CSV manually or add ratings here.


In [ ]:
eval_rows = []
for case in TEST_CASES:
    for model_name in ['sft', 'grpo']:
        if not (OUTPUT_DIR / model_name / f'{case["case_id"]}.png').exists():
            continue
        eval_rows.append({
            'case_id': case['case_id'],
            'group': case['group'],
            'model': model_name,
            'content_score_1_to_5': '',
            'style_score_1_to_5': '',
            'quality_score_1_to_5': '',
            'failure_tags': '',
            'notes': case['note'],
            'content_path': str(case['content_path']),
            'style_path': str(case['style_path']),
            'output_path': str(OUTPUT_DIR / model_name / f'{case["case_id"]}.png'),
        })

manual_eval = pd.DataFrame(eval_rows)
manual_eval_path = OUTPUT_DIR / 'manual_generalization_rubric.csv'
manual_eval.to_csv(manual_eval_path, index=False)
display(manual_eval)
print('saved:', manual_eval_path)


## 8. Optional: Broader Random Sweep

Use this only after the curated test works. It randomly pairs every content image with a few style images from selected categories.

This is useful for finding recurring failure patterns, but the curated cases above are better for presentation because each one has a reason.


In [ ]:
RUN_RANDOM_SWEEP = False
RANDOM_STYLES_PER_CONTENT = 3
RANDOM_STYLE_CATEGORIES = ['Sketch', 'WaterColor', 'OilPainting', 'PixelArt', 'LineDrawing', 'VanGogh', 'Monet', 'Cyberpunk']

if RUN_RANDOM_SWEEP:
    random.seed(123)
    random_cases = []
    content_files = sorted(CONTENT_DIR.glob('*.png'))
    for content_path in content_files:
        candidate_styles = []
        for category in RANDOM_STYLE_CATEGORIES:
            candidate_styles.extend(sorted((STYLE_DIR / category).glob('*.png')))
        selected_styles = random.sample(candidate_styles, min(RANDOM_STYLES_PER_CONTENT, len(candidate_styles)))
        for style_path in selected_styles:
            random_cases.append({
                'case_id': f'random_{content_path.stem}_{style_path.parent.name}_{style_path.stem}',
                'group': 'random_broad_sweep',
                'content_path': content_path,
                'style_path': style_path,
                'note': 'Random content/style pair for broad failure mining.',
            })

    if RUN_GRPO:
        _ = generate_checkpoint_outputs('grpo_random', ckpt_paths['grpo'], random_cases)
        make_comparison_grid(random_cases[:20], save_name='stylevar_random_sweep_preview_first20.png')


## 9. What To Conclude

After running, use this rule:

```text
If easier non-human cases are good but harder non-human cases fail:
    The paper's generalization limitation is confirmed on our images.

If both easier and harder cases are bad:
    Either the checkpoint/runtime loading is wrong, or StyleVAR is less robust than the paper examples suggest.

If GRPO is consistently better than SFT:
    The reward fine-tuning helps perceptual style/content trade-off.

If GRPO only improves style but hurts structure:
    GRPO may be pushing stylization strength at the cost of content preservation.
```

For our own VAR research, the important comparison is:

```text
StyleVAR trains the transformer to use content/style image conditions directly.
Our failed VAR-CLIP PFB/SAC experiment tried to inject image style without training that bridge.
```

So if StyleVAR works meaningfully better, the research lesson is not just "use StyleVAR." The lesson is that the missing part in pure VAR/VAR-CLIP experiments is probably the learned content/style conditioning bridge.
